<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="figures/cabecalho_branco.jpg" alt="Cabeçalho" style="max-width: 100%; height: auto;">
</div>

# Título (TODO)

**Autor**: Carlos Gabriel de Oliveira Campos

Professor Dr. **Daniel Roberto Cassar**

Disciplina de **Aprendizado de Máquina**

Introdução breve (TODO) 

---

# Constante Dielétrica

### Imports necessários para o notebook

In [297]:
from matminer.datasets import load_dataset
from matminer.featurizers.structure import DensityFeatures
from matminer.featurizers.composition import ElementProperty
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from optuna import create_study
from optuna.samplers import TPESampler
from functools import partial
import seaborn as sns
import numpy as np
import pandas as pd
import optuna

### Silenciamento de avisos

In [298]:
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning) # não-convergencia do lasso
optuna.logging.set_verbosity(optuna.logging.WARNING) # avisos gerais do optuna

### Importação do dataset e feature extraction

In [299]:
df = pd.read_pickle("dielectric_constant.pkl")
y = df.pop("poly_total")

In [300]:
# 2. Extrair atributos Estruturais (Densidade, volume por átomo, etc.)
dens_feat = DensityFeatures()
labels_dens = dens_feat.feature_labels()
features_dens = [dens_feat.featurize(struct) for struct in df["structure"]]
df_dens = pd.DataFrame(features_dens, columns=labels_dens, index=df.index)

# 3. Extrair atributos Composicionais (Magpie) a partir da estrutura
magpie_feat = ElementProperty.from_preset("magpie")
labels_magpie = magpie_feat.feature_labels()

# Pegamos a composição direto da estrutura (struct.composition)
features_magpie = [magpie_feat.featurize(struct.composition) for struct in df["structure"]]
df_magpie = pd.DataFrame(features_magpie, columns=labels_magpie, index=df.index)

# 4. Unir os dados nativos interessantes com as novas features
X = pd.concat([df, df_dens, df_magpie], axis=1)

print(f"\nMatriz gerada com sucesso! Linhas: {X.shape[0]}, Colunas: {X.shape[1]}")


Matriz gerada com sucesso! Linhas: 1056, Colunas: 150


### Processamento de coluna categórica

In [301]:
colunas_categoricas = ['pot_ferroelectric']

# Tudo que não estiver na lista acima será considerado numérico
colunas_numericas = [col for col in X.columns if col not in colunas_categoricas]

# 3. Criar o pré-processador dividido em dois caminhos paralelos
preprocessor = ColumnTransformer(
    transformers=[
        # Caminho 1: Padroniza APENAS as colunas quantitativas
        ('num', StandardScaler(), colunas_numericas),
        
        # Caminho 2: Aplica OHE em 'space_group' e 'pot_ferroelectric'.
        # O drop='if_binary' converte o booleano numa única coluna 0 ou 1.
        # NENHUMA dessas colunas passará pelo StandardScaler.
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), colunas_categoricas)
    ]
)

### Divisão treino-teste

In [302]:
TAMANHO_TESTE = 0.2
SEMENTE_ALEATORIA = 42

grupos = X['formula']

# 2. Configurar o separador de grupos
gss = GroupShuffleSplit(
    n_splits=1, 
    test_size=TAMANHO_TESTE,
    random_state=SEMENTE_ALEATORIA
)

for idx_treino, idx_teste in gss.split(X, y, groups=grupos):
    x_treino = X.iloc[idx_treino]
    x_teste = X.iloc[idx_teste]
    y_treino = y.iloc[idx_treino]
    y_teste = y.iloc[idx_teste]
    
    # Evita data leakage ao dividir pela formula das obs. de treino
    grupos_treino = grupos.iloc[idx_treino]


### Exclusão de atributos irrelevantes ou causados pelo target

Ex: `structure` é uma matriz de informações a partir da qual outras features são extraídas com as ferramentas do matminer. Já o índice de refração, `n`, está relacionado diretamente com o  target `poly_total`, visto que $n \approx \sqrt{\epsilon}$ (a conferir).

In [303]:
features_excluidas = ["cif", "material_id", "meta", "poscar", "structure", 
    "space_group", "formula", "e_electronic", "e_total", "n", "poly_electronic"]

X.drop(features_excluidas, axis=1, inplace=True);

### Baseline

In [304]:
baseline = DummyRegressor()
baseline.fit(x_treino, y_treino)
base_prev = baseline.predict(x_teste)

base_rmse = root_mean_squared_error(y_teste, base_prev)
base_r2 = r2_score(y_teste, base_prev)

base_rmse, base_r2

(16.525023435820987, -2.057879944228347e-06)

### Ridge

#### Instância do modelo

In [305]:
def instancia_ridge(trial):
    parametros = {
        # espaço de busca
        "alpha": trial.suggest_float("alpha", 0.01, 1_000, log=True)
    }

    modelo = Ridge(**parametros)

    return modelo

#### Função objetivo

In [312]:
def objetivo_ridge(trial, X, y, grupos, num_folds):
    """Função objetivo do optuna."""

    ridge_base = instancia_ridge(trial)

    metricas = []

    gkf = GroupKFold(num_folds)

    for passo, fold in enumerate(gkf.split(X, y, groups=grupos)):

        indice_treino, indice_teste = fold

        X_treino_, X_teste_ = X.iloc[indice_treino], X.iloc[indice_teste]
        y_treino_, y_teste_ = y.iloc[indice_treino], y.iloc[indice_teste]

        ridge = clone(ridge_base)
        ridge.fit(X_treino_, y_treino_)
        
        y_verdadeiro = y_teste_
        y_previsto = ridge.predict(X_teste_)

        RMSE = root_mean_squared_error(y_verdadeiro, y_previsto)

        metricas.append(RMSE)
        trial.report(np.mean(metricas), passo)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(metricas)

#### Aplicação do Optuna

In [313]:
NUM_TENTATIVAS = 100
NOME_DO_ESTUDO = "ridge_poda"

NUM_TRIALS_AQUECIMENTO = 5
NUM_PASSOS_AQUECIMENTO = 3

NUM_FOLDS = 5

PODADOR = optuna.pruners.MedianPruner(
    n_startup_trials=NUM_TRIALS_AQUECIMENTO,
    n_warmup_steps=NUM_PASSOS_AQUECIMENTO,
)

estudo_ridge = create_study(
    direction="minimize",
    study_name=NOME_DO_ESTUDO,
    storage=f"sqlite:///{NOME_DO_ESTUDO}.db",
    load_if_exists=True,
    sampler=TPESampler(multivariate=True),
    pruner=PODADOR,
)

parcial_ridge = partial(
    objetivo_ridge,
    X=X,
    y=y,
    grupos=grupos,
    num_folds=NUM_FOLDS
)

estudo_ridge.optimize(parcial_ridge, n_trials=NUM_TENTATIVAS)

c:\venv\ilumpy\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


In [315]:
estudo_ridge.best_trial

FrozenTrial(number=7, state=1, values=[17.603182095417885], datetime_start=datetime.datetime(2026, 10, 15, 17, 28, 28, 653444), datetime_complete=datetime.datetime(2026, 10, 15, 17, 28, 28, 974182), params={'alpha': 62.33462238885137}, user_attrs={}, system_attrs={}, intermediate_values={0: 22.96592907041477, 1: 17.72192808555424, 2: 17.235067286392834, 3: 17.059597274733225, 4: 17.603182095417885}, distributions={'alpha': FloatDistribution(high=1000.0, log=True, low=0.01, step=None)}, trial_id=8, value=None)

### Lasso (TODO)

### Floresta Aleatória (TODO)

In [ ]:
def cria_instancia_modelo(trial):
    """Cria uma instância do modelo.

    Args:
      trial: objeto tipo Trial do optuna.

    Returns:
      Uma instância do modelo desejado.
    """

    parametros = {
        # espaço de busca
        "n_estimators": trial.suggest_int("num_arvores", 5, 300, log=True),
        "criterion": trial.suggest_categorical("critério", ["squared_error", "absolute_error", "poisson"]),
        "min_samples_split": trial.suggest_int("min_exemplos_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_exemplos_folha", 1, 20),
        "max_features": trial.suggest_float("num_max_atributos", 0, 1),
        # argumentos fixos
        "n_jobs": -1,
        "bootstrap": True,
        "random_state": SEMENTE_ALEATORIA,
    }

    modelo = RandomForestRegressor(**parametros)

    return modelo

## Conclusão

## Referências

## Uso de IA neste trabalho

<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="figures/rodape_eleicao.png" alt="Rodapé" style="max-width: 100%; height: auto;">
</div>